# 📌 Analisis Sentimen Masyarakat terhadap Proyek Transportasi Modern di Indonesia melalui Komentar YouTube


## 1. Business Understanding
### Latar Belakang

Infrastruktur transportasi modern seperti Kereta Cepat Whoosh, MRT Jakarta, dan LRT Jabodebek merupakan bagian dari transformasi sistem transportasi nasional. Kehadiran proyek-proyek tersebut memunculkan berbagai respons dari masyarakat yang dapat diamati melalui media sosial, salah satunya YouTube.

Komentar pengguna YouTube mengandung opini, pengalaman, serta persepsi masyarakat terhadap kualitas layanan, manfaat ekonomi, kenyamanan, hingga efektivitas pembangunan transportasi modern.

Analisis sentimen dapat digunakan untuk mengidentifikasi kecenderungan opini masyarakat secara otomatis menggunakan teknik Natural Language Processing (NLP).

### Tujuan
1. Mengumpulkan komentar publik dari YouTube.
2. Mengklasifikasikan sentimen menjadi positif, netral, dan negatif.
3. Membangun model machine learning dan deep learning.
4. Membandingkan performa beberapa algoritma.

## 2. Import Library

### Tujuan

Menginstal library yang diperlukan untuk scraping data.


In [ ]:
!pip install google-api-python-client pandas tqdm

In [ ]:
!pip install pipreqs

In [ ]:
!pip freeze > requirements.txt

## 3. Koneksi ke YouTube Data API
### Tujuan

Menghubungkan notebook dengan YouTube Data API v3 agar dapat melakukan pencarian video dan pengambilan komentar.

## Import Library

In [ ]:
import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build

## Koneksi YouTube API

In [ ]:
from google.colab import userdata
from googleapiclient.discovery import build

API_KEY = userdata.get("YOUTUBE_API_KEY")

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY
)

print("API Connected")

API Connected


In [ ]:
request = youtube.videos().list(
    part="snippet",
    chart="mostPopular",
    regionCode="ID",
    maxResults=1
)

response = request.execute()

print("YouTube API request successful")
print("Video title:", response["items"][0]["snippet"]["title"])

YouTube API request successful
Video title: Dermaga Terakhir Di Hatimu


## 4. Validasi API

### Tujuan

Memastikan API dapat digunakan sebelum proses scraping dilakukan.

## 5. Menentukan Kata Kunci Pencarian

###Tujuan
Mengumpulkan video yang berkaitan dengan Infrastruktur transportasi modern seperti Kereta Cepat Whoosh, MRT Jakarta, dan LRT Jabodebek. Keyword dipilih berdasarkan kata-kata yang populer.

In [ ]:
request = youtube.search().list(
    q="kereta cepat whoosh",
    part="snippet",
    maxResults=5,
    type="video"
)

response = request.execute()

print("Jumlah video:", len(response["items"]))

Jumlah video: 5


## Menentukan Kata Kunci

In [ ]:
video_ids = []

keywords = [

    # Whoosh
    "kereta cepat jakarta bandung",
    "kereta cepat whoosh",
    "review whoosh",
    "naik whoosh",

    # MRT
    "mrt jakarta",
    "review mrt jakarta",
    "naik mrt jakarta",
    "pengalaman mrt jakarta",

    # LRT
    "lrt jabodebek",
    "review lrt jabodebek",
    "naik lrt jabodebek",
    "pengalaman lrt jabodebek"

]

keywords.extend([

    "kereta cepat indonesia",

    "kereta cepat whoosh terbaru",

    "whoosh cnbc",

    "whoosh kompas tv",

    "mrt jakarta terbaru",

    "lrt jabodebek terbaru",

    "review kereta cepat",

    "pengalaman naik whoosh"

])

##6. Mengumpulkan Daftar Video

###Tujuan
Mengambil video yang relevan berdasarkan kata kunci yang telah ditentukan.

In [ ]:
video_ids = []

for keyword in keywords:

    print("Keyword:", keyword)

    request = youtube.search().list(
        q=keyword,
        part="snippet",
        maxResults=10,
        type="video"
    )

    response = request.execute()

    for item in response["items"]:

        if "videoId" in item["id"]:

            video_ids.append(
                item["id"]["videoId"]
            )

video_ids = list(set(video_ids))

print("Jumlah video unik:", len(video_ids))

Keyword: kereta cepat jakarta bandung
Keyword: kereta cepat whoosh
Keyword: review whoosh
Keyword: naik whoosh
Keyword: mrt jakarta
Keyword: review mrt jakarta
Keyword: naik mrt jakarta
Keyword: pengalaman mrt jakarta
Keyword: lrt jabodebek
Keyword: review lrt jabodebek
Keyword: naik lrt jabodebek
Keyword: pengalaman lrt jabodebek
Keyword: kereta cepat indonesia
Keyword: kereta cepat whoosh terbaru
Keyword: whoosh cnbc
Keyword: whoosh kompas tv
Keyword: mrt jakarta terbaru
Keyword: lrt jabodebek terbaru
Keyword: review kereta cepat
Keyword: pengalaman naik whoosh
Jumlah video unik: 153


## 7. Scraping Komentar dan Evaluasi Hasil Scraping
###Tujuan

Mengambil komentar dari seluruh video yang berhasil dikumpulkan. Komentar inilah yang nantinya akan digunakan sebagai dataset analisis sentimen. Memastikan jumlah komentar telah memenuhi kebutuhan dataset dan sesuai dengan ketentuan submission.

In [ ]:
import pandas as pd
import numpy as np

comments = []

from tqdm import tqdm

for video_id in tqdm(video_ids):

    try:

        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100
        )

        response = request.execute()

        while True:

            for item in response["items"]:

                comment = item[
                    "snippet"
                ]["topLevelComment"][
                    "snippet"
                ]["textDisplay"]

                comments.append(comment)

            if "nextPageToken" in response:

                request = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    maxResults=100,
                    pageToken=response["nextPageToken"]
                )

                response = request.execute()

            else:
                break

    except:
        pass

print(
    "Jumlah komentar:",
    len(comments)
)

100%|██████████| 153/153 [00:27<00:00,  5.63it/s]

Jumlah komentar: 32712


## 8. Membentuk Dataset

###Tujuan
Mengubah seluruh komentar menjadi DataFrame agar bisa dianalisis. Dataset hasil scraping akan disimpan dalam bentuk DataFrame dengan satu kolom yaitu "Comment" yang berisi komentar pengguna YouTube terkait Infrastruktur transportasi modern seperti Kereta Cepat Whoosh, MRT Jakarta, dan LRT Jabodebek.

In [ ]:
import pandas as pd

df = pd.DataFrame(
    comments,
    columns=["comment"]
)

print(df.shape)

df.head()

(32712, 1)


,comment
0,Dana MBG 235 trilyun setahun. Itu lebih dari c...
1,Gak jalan 😂 sama kayak hambalang
2,Ahy mendukung wosh sama saja membunuh menguran...
3,Kemampuan menteri dan era kepemimpinan sekaran...
4,Ga bakal jadi klo AHY yg urus infrastruktur. D...


##9. Data Understanding
### Tujuan
Melihat kualitas data hasil scraping. Tahap ini dilakukan untuk:
1. Memastikan tidak ada nilai kosong
2. Mengetahui tipe data
3. Memastikan seluruh komentar berhasil tersimpan



In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32712 entries, 0 to 32711
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   comment  32712 non-null  object
dtypes: object(1)
memory usage: 255.7+ KB


In [ ]:
df.isnull().sum()

,0
comment,0


##10. Menghapus Duplikasi & Filtering Komentar Tidak Informatif
###Tujuan
Menghilangkan komentar yang sama agar model tidak bias.

Tahap ini juga bertujuan menghapus komentar yang terlalu pendek atau terlalu panjang sehingga tidak memberikan informasi sentimen yang memadai.

Komentar yang terdiri dari emoji, simbol, atau hanya satu kata umumnya tidak memberikan konteks yang cukup untuk proses analisis sentimen.

In [ ]:
df = df.drop_duplicates()

print(df.shape)

(32004, 1)


In [ ]:
df["comment_length"] = df["comment"].astype(str).apply(len)

df["comment_length"].describe()

,comment_length
count,32004.00000
mean,99.06068
std,195.95278
min,1.00000
25%,31.00000
50%,59.00000
75%,112.00000
max,10420.00000


### Insight

Proses filtering panjang komentar berhasil mengurangi data yang tidak informatif tanpa menghilangkan sebagian besar dataset.

Jumlah komentar berkurang dari 32.712 menjadi 32.004 komentar, atau hanya sekitar 2,16% dari total data. Hal ini menunjukkan bahwa mayoritas komentar yang berhasil dikumpulkan memiliki panjang yang cukup untuk merepresentasikan opini atau sentimen pengguna.

Dataset hasil filtering dinilai memiliki kualitas yang lebih baik karena telah mengurangi komentar berupa emoji, simbol, singkatan yang terlalu pendek, maupun komentar yang sangat panjang dan berpotensi mengandung noise.

##11. Simpan Dataset Mentah
###Tujuan
Menyimpan hasil scraping sebagai backup apabila quota API habis atau runtime Colab ter-reset.

In [ ]:
df.to_csv(
    "dataset_transportasi_raw.csv",
    index=False
)

print("Dataset berhasil disimpan.")

Dataset berhasil disimpan.


In [ ]:
### END OF SCRAPING ###